# Mini LLM Inference Engine — run on Colab

This notebook only *runs* the project. All the code lives in the Python package in the repo.

1. **Runtime → Change runtime type → T4 GPU** (the project is designed for a T4).
2. **Runtime → Run all.**

Every cell is safe to run again after a disconnect: the setup cell clones the repo only once and pulls on later runs.

In [ ]:
# Setup (idempotent): clone or update the repo, then install it.
import os

REPO_URL = "https://github.com/Ham12-3/mini-inference.git"
REPO_DIR = "/content/mini-inference"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
# This checkout is a read-only mirror of GitHub: benchmark runs write result
# files into it, so update with fetch + hard reset rather than pull, which
# would refuse to overwrite them.
!git fetch -q && git reset -q --hard origin/master
!git log --oneline -1

# Lower-bound requirements only, so Colab's preinstalled torch is reused.
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## Stage 0: GPU check and model config

In [ ]:
!nvidia-smi
!python -m benchmarks.env_info --model Qwen/Qwen2.5-0.5B-Instruct

## Stage 1: naive baseline (no KV cache)

Every new token re-runs the whole sequence through the model. About 2.5 minutes.

In [ ]:
!python -m benchmarks.single_request --backend naive --lengths 64 128 256 --repeats 3
!python -m benchmarks.single_request --backend naive --lengths 512 1024 --repeats 1 --name single_naive_long

## Stage 2: my Qwen2 model with a KV cache

Parity tests against Hugging Face, then single-request speed for my engine and for HF `generate()`.

In [ ]:
!python -m pytest tests/test_model_parity.py -q -p no:warnings

In [ ]:
!python -m benchmarks.single_request --backend engine_kv --lengths 64 128 256 512 1024 --repeats 3
!python -m benchmarks.single_request --backend hf_kv --lengths 64 128 256 512 1024 --repeats 3

In [ ]:
!python -m benchmarks.plot
from IPython.display import Image, display
display(Image('benchmarks/results/chart1_single_request.png'))